In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import pandas as pd
import os
import sys
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm import tqdm
import warnings
import shutil
import time

# 自动尝试安装 EduData
try:
    from EduData import get_data
except ImportError:
    print("正在安装 EduData...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "EduData"])
    from EduData import get_data

warnings.filterwarnings('ignore')

# =========================================
# 1. 基础配置 (Configuration)
# =========================================

class Config:
    def __init__(self):
        self.num_students = 0
        self.num_questions = 0
        self.num_concepts = 0

        # 模型超参数
        self.embed_dim = 128
        self.seq_len = 100
        self.tcan_layers = 3
        self.num_heads = 8
        self.kernel_size = 3
        self.dropout = 0.2

        # 训练参数
        self.batch_size = 128
        self.epochs = 10
        self.lr = 0.001
        self.weight_decay = 1e-5
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        # 数据路径
        self.data_dir = "./data/junyi"
        self.dataset_name = "junyi"
        self.max_rows = 500000

        # --- 消融实验配置 (Ablation Settings) ---
        self.use_hg = True       # 是否使用异构图嵌入 (Hetero-Graph)
        self.use_diff = True     # 是否在图中加入难度特征 (Difficulty)
        self.use_tcan = True     # 是否使用 TCAN 块 (若为 False 则退化为普通 Linear)
        self.use_res_emb = True  # 是否使用响应嵌入 (Response Embedding)

# =========================================
# 2. 数据处理模块 (Data Processing)
# =========================================

class JunyiDataset(Dataset):
    def __init__(self, q_ids, labels, s_ids, max_len):
        self.q_ids = q_ids
        self.labels = labels
        self.s_ids = s_ids
        self.max_len = max_len

    def __len__(self):
        return len(self.q_ids)

    def __getitem__(self, idx):
        q_seq = self.q_ids[idx]
        r_seq = self.labels[idx]
        s_id = self.s_ids[idx]
        seq_len = len(q_seq)

        if seq_len >= self.max_len:
            q_seq = q_seq[-self.max_len:]
            r_seq = r_seq[-self.max_len:]
            mask = np.ones(self.max_len, dtype=np.float32)
        else:
            pad_len = self.max_len - seq_len
            q_seq = np.pad(q_seq, (pad_len, 0), 'constant', constant_values=0)
            r_seq = np.pad(r_seq, (pad_len, 0), 'constant', constant_values=0)
            mask = np.concatenate([np.zeros(pad_len), np.ones(seq_len)]).astype(np.float32)

        return (
            torch.tensor(q_seq, dtype=torch.long),
            torch.tensor(r_seq, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32),
            torch.tensor(s_id, dtype=torch.long)
        )

class DataProcessor:
    def __init__(self, config):
        self.config = config

    def _find_data_file(self):
        if not os.path.exists(self.config.data_dir): return None
        for root, _, files in os.walk(self.config.data_dir):
            for file in files:
                if "problemlog" in file.lower() and file.lower().endswith(".csv"):
                    return os.path.join(root, file)
        return None

    def process(self):
        file_path = self._find_data_file()
        if not file_path:
            get_data(self.config.dataset_name, self.config.data_dir)
            file_path = self._find_data_file()

        print(f"正在读取数据: {file_path}")
        df = pd.read_csv(file_path, nrows=self.config.max_rows)

        rename_map = {}
        found_targets = set()
        for col in df.columns:
            low = col.lower()
            target = None
            if ('user' in low and 'id' in low) or low == 'user_id': target = 'user_id'
            elif low == 'exercise' or 'problem' in low: target = 'item_id'
            elif 'topic' in low or 'skill' in low: target = 'skill_id'
            elif 'correct' in low: target = 'correct'
            elif 'timestamp' in low or (target is None and 'time' in low): target = 'timestamp'
            if target and target not in found_targets:
                rename_map[col] = target
                found_targets.add(target)

        df.rename(columns=rename_map, inplace=True)
        df = df.loc[:, ~df.columns.duplicated()]

        if 'skill_id' not in df.columns: df['skill_id'] = df['item_id']
        df['correct'] = df['correct'].apply(lambda x: 1 if str(x).lower() in ['1', '1.0', 'true'] else 0)
        df.dropna(subset=['user_id', 'item_id', 'skill_id'], inplace=True)

        if 'timestamp' in df.columns:
            if isinstance(df['timestamp'], pd.DataFrame): df['timestamp'] = df['timestamp'].iloc[:, 0]
            df.sort_values(by=['user_id', 'timestamp'], inplace=True)

        user_map = {val: i+1 for i, val in enumerate(df['user_id'].unique())}
        item_map = {val: i+1 for i, val in enumerate(df['item_id'].unique())}
        skill_map = {val: i+1 for i, val in enumerate(df['skill_id'].unique())}

        df['u_idx'] = df['user_id'].map(user_map)
        df['i_idx'] = df['item_id'].map(item_map)
        df['s_idx'] = df['skill_id'].map(skill_map)

        self.config.num_students = len(user_map) + 1
        self.config.num_questions = len(item_map) + 1
        self.config.num_concepts = len(skill_map) + 1

        item_diff = df.groupby('i_idx')['correct'].mean()
        diff_values = np.zeros(self.config.num_questions)
        for iid, diff in item_diff.items(): diff_values[iid] = diff

        q_k_rel = np.zeros((self.config.num_questions, self.config.num_concepts))
        tmp = df[['i_idx', 's_idx']].drop_duplicates()
        for _, row in tmp.iterrows(): q_k_rel[int(row['i_idx']), int(row['s_idx'])] = 1
        row_sum = q_k_rel.sum(axis=1, keepdims=True)
        q_k_rel = np.divide(q_k_rel, row_sum, out=np.zeros_like(q_k_rel), where=row_sum!=0)

        all_q, all_r, all_s = [], [], []
        for _, group in tqdm(df.groupby('u_idx'), desc="构造序列"):
            if len(group) < 3: continue
            q_list, r_list = group['i_idx'].values, group['correct'].values
            u_id = group['u_idx'].iloc[0]
            for i in range(0, len(q_list), self.config.seq_len):
                sub_q, sub_r = q_list[i:i + self.config.seq_len], r_list[i:i + self.config.seq_len]
                if len(sub_q) < 3: continue
                all_q.append(sub_q); all_r.append(sub_r); all_s.append(u_id)

        return all_q, all_r, all_s, q_k_rel, diff_values

# =========================================
# 3. 核心模型 (支持消融实验)
# =========================================

class HeteroGraphEmbedding(nn.Module):
    def __init__(self, config, diff_values):
        super().__init__()
        self.config = config
        self.q_emb = nn.Embedding(config.num_questions, config.embed_dim, padding_idx=0)
        self.c_emb = nn.Embedding(config.num_concepts, config.embed_dim, padding_idx=0)
        self.register_buffer('diff', torch.tensor(diff_values, dtype=torch.float32))
        self.diff_proj = nn.Linear(1, config.embed_dim)

        # 动态计算输入维度
        input_dim = config.embed_dim
        if config.use_hg: input_dim += config.embed_dim
        if config.use_diff: input_dim += config.embed_dim

        self.fusion = nn.Sequential(
            nn.Linear(input_dim, config.embed_dim),
            nn.LayerNorm(config.embed_dim),
            nn.ReLU(),
            nn.Dropout(config.dropout)
        )

    def forward(self, q_ids, q_k_rel):
        q_f = self.q_emb(q_ids)
        feats = [q_f]

        if self.config.use_hg:
            k_weights = self.c_emb.weight
            c_f = torch.matmul(q_k_rel, k_weights)
            c_f_expanded = F.embedding(q_ids, c_f, padding_idx=0)
            feats.append(c_f_expanded)

        if self.config.use_diff:
            d_f = self.diff_proj(self.diff[q_ids].unsqueeze(-1))
            feats.append(d_f)

        combined = torch.cat(feats, dim=-1)
        return self.fusion(combined)

class TCANBlock(nn.Module):
    def __init__(self, dim, num_heads, k, dila, drop):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads=num_heads, dropout=drop, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.conv = nn.Conv1d(dim, dim, k, dilation=dila, padding=(k-1)*dila)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        nx = self.norm1(x)
        mask = torch.triu(torch.ones(x.size(1), x.size(1), device=x.device), diagonal=1).bool()
        attn_out, _ = self.attn(nx, nx, nx, need_weights=False, attn_mask=mask)
        x = x + attn_out

        nx = self.norm2(x)
        conv_in = nx.transpose(1, 2)
        conv_out = self.conv(conv_in)[:, :, :x.size(1)].transpose(1, 2)
        x = x + self.drop(F.relu(conv_out))
        return x

class HIG_TCAN_Model(nn.Module):
    def __init__(self, config, q_k_rel, diffs):
        super().__init__()
        self.config = config
        self.register_buffer('q_k_rel', torch.tensor(q_k_rel, dtype=torch.float32))
        self.graph_emb = HeteroGraphEmbedding(config, diffs)

        self.num_q = config.num_questions
        # 只有在使用 ResEmb 时才初始化
        if config.use_res_emb:
            self.res_emb = nn.Embedding(config.num_questions * 2 + 1, config.embed_dim, padding_idx=0)
        else:
            self.q_only_emb = nn.Embedding(config.num_questions, config.embed_dim - 1, padding_idx=0)

        self.stu_emb = nn.Embedding(config.num_students, config.embed_dim, padding_idx=0)

        if config.use_tcan:
            self.tcan = nn.ModuleList([
                TCANBlock(config.embed_dim, config.num_heads, config.kernel_size, 2**i, config.dropout)
                for i in range(config.tcan_layers)
            ])
        else:
            self.simple_proj = nn.Linear(config.embed_dim, config.embed_dim)

        self.out = nn.Sequential(
            nn.Linear(config.embed_dim * 3, config.embed_dim),
            nn.LayerNorm(config.embed_dim),
            nn.ReLU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.embed_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, q, r, s):
        if self.config.use_res_emb:
            res_idx = q + (r.long() * self.num_q)
            res_idx = res_idx * (q > 0).long()
            x = self.res_emb(res_idx)
        else:
            # 消融实验：不使用 Response Embedding，退化为基础拼接
            q_base = self.q_only_emb(q)
            x = torch.cat([q_base, r.unsqueeze(-1)], dim=-1)

        h = x
        if self.config.use_tcan:
            for layer in self.tcan: h = layer(h)
        else:
            h = F.relu(self.simple_proj(h))

        q_feat = self.graph_emb(q, self.q_k_rel)
        s_feat = self.stu_emb(s).unsqueeze(1).expand(-1, q.size(1), -1)

        return h, q_feat, s_feat

# =========================================
# 4. 训练与消融实验循环
# =========================================

def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for q, r, m, s in loader:
            q, r, m, s = q.to(device), r.to(device), m.to(device), s.to(device)
            h, q_f, s_f = model(q, r, s)
            feat = torch.cat([h[:, :-1, :], q_f[:, 1:, :], s_f[:, 1:, :]], dim=-1)
            p = model.out(feat).squeeze(-1)
            mask = m[:, 1:].bool()
            if mask.sum() == 0: continue
            y_pred.append(p[mask].cpu().numpy())
            y_true.append(r[:, 1:][mask].cpu().numpy())
    return roc_auc_score(np.concatenate(y_true), np.concatenate(y_pred)), \
           accuracy_score(np.concatenate(y_true), np.concatenate(y_pred) > 0.5)

def train_one_variant(variant_name, cfg, data):
    q, r, s, qk, diff = data
    t_q, v_q, t_r, v_r, t_s, v_s = train_test_split(q, r, s, test_size=0.2, random_state=42)
    train_ld = DataLoader(JunyiDataset(t_q, t_r, t_s, cfg.seq_len), batch_size=cfg.batch_size, shuffle=True)
    val_ld = DataLoader(JunyiDataset(v_q, v_r, v_s, cfg.seq_len), batch_size=cfg.batch_size)

    model = HIG_TCAN_Model(cfg, qk, diff).to(cfg.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    print(f"\n>>> 实验变体: {variant_name} (HG={cfg.use_hg}, Diff={cfg.use_diff}, TCAN={cfg.use_tcan})")

    best_auc = 0
    for ep in range(cfg.epochs):
        model.train()
        for q_b, r_b, m_b, s_b in train_ld:
            q_b, r_b, m_b, s_b = q_b.to(cfg.device), r_b.to(cfg.device), m_b.to(cfg.device), s_b.to(cfg.device)
            optimizer.zero_grad()
            h, q_f, s_f = model(q_b, r_b, s_b)
            feat = torch.cat([h[:, :-1, :], q_f[:, 1:, :], s_f[:, 1:, :]], dim=-1)
            p = model.out(feat).squeeze(-1)
            loss = F.binary_cross_entropy(p, r_b[:, 1:], reduction='none')
            loss = (loss * m_b[:, 1:]).sum() / (m_b[:, 1:].sum() + 1e-8)
            loss.backward()
            optimizer.step()

        auc, acc = evaluate(model, val_ld, cfg.device)
        if auc > best_auc: best_auc = auc
        print(f"Epoch {ep+1}/{cfg.epochs} | AUC: {auc:.4f} | ACC: {acc:.4f}")

    return best_auc

def run_ablation_study():
    cfg = Config()
    proc = DataProcessor(cfg)
    data = proc.process()

    results = {}

    # 1. Full Model
    results['HIG-TCAN (Full)'] = train_one_variant('Full Model', cfg, data)

    # 2. w/o Hetero-Graph
    cfg.use_hg = False
    results['w/o HG'] = train_one_variant('w/o HG', cfg, data)
    cfg.use_hg = True # Reset

    # 3. w/o Difficulty
    cfg.use_diff = False
    results['w/o Diff'] = train_one_variant('w/o Diff', cfg, data)
    cfg.use_diff = True # Reset

    # 4. w/o TCAN
    cfg.use_tcan = False
    results['w/o TCAN'] = train_one_variant('w/o TCAN', cfg, data)
    cfg.use_tcan = True # Reset

    print("\n" + "="*30)
    print("消融实验最终结果 (Best AUC):")
    for k, v in results.items():
        print(f"{k}: {v:.4f}")
    print("="*30)

if __name__ == "__main__":
    run_ablation_study()

正在安装 EduData...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.2/141.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 MB 24.0 MB/s eta 0:00:00
  Attempting uninstall: typeguard
    Found existing installation: typeguard 4.4.4
    Uninstalling typeguard-4.4.4:
      Successfully uninstalled typeguard-4.4.4
  Attempting uninstall: filelock
    Found existing installation: filelock 3.20.1
    Uninstalling filelock-3.20.1:
      Successfully uninstalled filelock-3.20.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kauldron 1.3.0 requires typeguard>=4.4.1, but you have typeguard 4.1.2 which is incompatible.
ray 2.52.1 requires click!=8.3.*,>=7.0, but you have click 8.3.1 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.
pytensor 2.35.1 requires filelock>=3.15, but you have filelock 3.11.0 which is incompatible.


downloader, INFO http://base.ustc.edu.cn/data/JunyiAcademy_Math_Practicing_Log/junyi.rar is saved as data/junyi/junyi.rar


downloader, INFO data/junyi/junyi.rar is unrar to data/junyi/junyi



正在读取数据: ./data/junyi/junyi/junyi_ProblemLog_original.csv


构造序列: 100%|██████████| 87502/87502 [00:04<00:00, 21352.14it/s]



>>> 实验变体: Full Model (HG=True, Diff=True, TCAN=True)
Epoch 1/10 | AUC: 0.7173 | ACC: 0.8340
Epoch 2/10 | AUC: 0.7212 | ACC: 0.8353
Epoch 3/10 | AUC: 0.7211 | ACC: 0.8354
Epoch 4/10 | AUC: 0.7155 | ACC: 0.8339
Epoch 5/10 | AUC: 0.7063 | ACC: 0.8325
Epoch 6/10 | AUC: 0.6913 | ACC: 0.8280
Epoch 7/10 | AUC: 0.6799 | ACC: 0.8264
Epoch 8/10 | AUC: 0.6603 | ACC: 0.8182
Epoch 9/10 | AUC: 0.6491 | ACC: 0.7985
Epoch 10/10 | AUC: 0.6360 | ACC: 0.7952

>>> 实验变体: w/o HG (HG=False, Diff=True, TCAN=True)
Epoch 1/10 | AUC: 0.7186 | ACC: 0.8342
Epoch 2/10 | AUC: 0.7230 | ACC: 0.8357
Epoch 3/10 | AUC: 0.7214 | ACC: 0.8364
Epoch 4/10 | AUC: 0.7147 | ACC: 0.8349
Epoch 5/10 | AUC: 0.7086 | ACC: 0.8321
Epoch 6/10 | AUC: 0.6989 | ACC: 0.8215
Epoch 7/10 | AUC: 0.6841 | ACC: 0.8260
Epoch 8/10 | AUC: 0.6729 | ACC: 0.8181
Epoch 9/10 | AUC: 0.6530 | ACC: 0.8107
Epoch 10/10 | AUC: 0.6491 | ACC: 0.8025

>>> 实验变体: w/o Diff (HG=True, Diff=False, TCAN=True)
Epoch 1/10 | AUC: 0.7094 | ACC: 0.8332
Epoch 2/10 | AUC: 0.7